# HKRL 一键克隆与训练环境配置（Kaggle / Conda 双后端）

这份 Notebook 可以独立上传到 Kaggle 或普通 Linux/NVIDIA GPU 训练机。修改参数格、把 `RUN_SETUP=True`，然后选择 **Run All**，即可完成：

1. 检测 Kaggle、GPU、CUDA、网络、磁盘和可写目录；
2. 克隆或安全复用 HKRL 仓库；
3. Kaggle 使用当前 Python/Pip，普通训练机使用 Conda；
4. 安装 HKRL 及分布式依赖；
5. 验证训练配置和 GPU Learner；
6. Kaggle 用合成 rollout 执行一次 GPU 优化器更新，再运行 Phase 8 离线 smoke 并打包结果；
7. 普通训练机创建权限为 `0600` 的认证令牌；
8. 写入不含密钥的安装摘要。

> 默认是预览模式，不克隆、不安装、不更新、不覆盖。已有目标目录绝不会被删除。

## Goal

### Kaggle 能验证什么

- 当前 PyTorch 是否能看到 CUDA GPU；
- HKRL 模型、配置、Learner 和 checkpoint 是否能在 GPU 环境初始化；
- 一份明确标记为非游戏数据的合成 rollout 能否完成 forward/backward/optimizer 更新；
- worker/learner/coordinator 的 Phase 8 离线连接关系；
- checkpoint 是否可生成、加载和导出。

### Kaggle 不能替代什么

Kaggle 没有 Hollow Knight、Mod 和 Windows 本地 GameWorker，也不适合作为需要 Windows 主动长连的 SSH 服务端。因此它不能产生真实游戏 rollout、胜率或最终模型能力结论。真实分布式训练仍按 `docs/windows_ssh_deployment.md` 在可 SSH 到达的 GPU 主机上运行。

Kaggle 官方 GPU 镜像本身持续更新；Notebook 会检测当前实际的 `torch`/CUDA，而不是假定固定版本。

## Setup

### 0. Kaggle 页面设置

在 Kaggle Notebook 右侧 **Settings** 中：

1. 将 **Accelerator** 设为 GPU；
2. 将 **Internet** 设为 On，供 Git clone 和缺失依赖安装；
3. 上传本 Notebook，然后先用预览模式 Run All。

Kaggle 官方文档建议只在实际使用 GPU 时开启加速器并监控配额；官方 CLI 文档也列出了 Notebook 的 GPU accelerator 选项。

### 1. 只修改这一格

- 第一次保持 `RUN_SETUP=False` 查看检查结果。
- Kaggle 通常只需把 `RUN_SETUP=True`，其他参数保持默认。
- `ENV_BACKEND="auto"`：Kaggle/无 Conda → `current_python`；其他训练机 → `conda`。
- Kaggle 默认复用预装 Torch；只有明确诊断为 Torch 构建不兼容时才设置 `REINSTALL_TORCH=True`。

In [ ]:
from pathlib import Path

RUN_SETUP = False

REPO_URL = "https://github.com/myouo/hk_Rl.git"
BRANCH = "main"
TARGET_DIR = None  # auto: Kaggle=/kaggle/working/hk_Rl，其他=~/hk_Rl

ENV_BACKEND = "auto"  # auto | current_python | conda
CONDA_ENV = "hkrl-learner"
CONDA_EXE = ""  # Conda 后端可填 /opt/conda/bin/conda
TORCH_INDEX_URL = ""  # 仅重装 Torch 时填写官方 HTTPS wheel index
REINSTALL_TORCH = False
ALLOW_CPU = False  # Kaggle GPU smoke 保持 False

UPDATE_EXISTING = False  # True 时只允许 clean worktree + fast-forward
CREATE_AUTH_TOKEN = None  # auto: Kaggle=False，普通训练机=True
TOKEN_FILE = None
CHECK_NETWORK = True
RUN_KAGGLE_SMOKE = True
RUN_PROJECT_TESTS = False
IS_KAGGLE_OVERRIDE = None  # 仅调试时使用 True/False

print({
    "run_setup": RUN_SETUP,
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "target_dir": None if TARGET_DIR is None else str(TARGET_DIR),
    "env_backend": ENV_BACKEND,
    "update_existing": UPDATE_EXISTING,
    "allow_cpu": ALLOW_CPU,
    "run_kaggle_smoke": RUN_KAGGLE_SMOKE,
})

### Key Assumptions

- Kaggle 使用预装的当前 Python/PyTorch，不需要也不创建 Conda 环境。
- NVIDIA 驱动由 Kaggle/主机提供，本 Notebook 不修改内核驱动。
- Kaggle 的持久输出应放在 `/kaggle/working`；`/kaggle/input` 视为只读输入。
- 目标目录若已存在，必须是同一 `origin`、同一分支的 Git 仓库。
- 更新已有仓库时，工作区必须完全干净；只执行 `fetch + merge --ff-only`。
- Kaggle smoke 不创建认证令牌；普通训练机令牌不会显示在输出或 manifest 中。
- Pip 安装后所有验证都在新的子进程中执行，避免当前 Notebook kernel 的旧模块缓存。

## Steps

### 2. 环境预检

本格只读取环境状态。网络探测失败不会在预览模式中止，但执行模式会给出明确修复提示。

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import re
import secrets
import shlex
import shutil
import stat
import subprocess
import sys
from datetime import datetime, timezone
from urllib.parse import urlparse
from urllib.request import Request, urlopen


def normalized_repo_url(value: str) -> str:
    return value.strip().rstrip("/").removesuffix(".git")


def resolve_program(override: str, name: str) -> str:
    if override.strip():
        candidate = Path(override).expanduser()
        if not candidate.is_file():
            raise FileNotFoundError(f"{name} 不存在: {candidate}")
        return str(candidate.resolve())
    discovered = shutil.which(name)
    return discovered or ""


def probe_url(url: str) -> dict[str, object]:
    request = Request(url, headers={"User-Agent": "hkrl-setup-check/1"})
    try:
        with urlopen(request, timeout=5.0) as response:
            return {"ok": True, "status": int(response.status)}
    except Exception as exc:
        return {"ok": False, "error": f"{type(exc).__name__}: {exc}"}


repo_url = REPO_URL.strip()
branch = BRANCH.strip()
detected_kaggle = bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE")
    or (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir())
)
is_kaggle = detected_kaggle if IS_KAGGLE_OVERRIDE is None else bool(IS_KAGGLE_OVERRIDE)
workspace_root = Path("/kaggle/working") if is_kaggle else Path.home()
target_dir = (
    (workspace_root / "hk_Rl").resolve()
    if TARGET_DIR is None
    else Path(TARGET_DIR).expanduser().resolve()
)
create_auth_token = (
    not is_kaggle if CREATE_AUTH_TOKEN is None else bool(CREATE_AUTH_TOKEN)
)
token_file = (
    (Path("/tmp/hkrl-kaggle/auth_token") if is_kaggle else Path.home() / ".config/hkrl/auth_token")
    if TOKEN_FILE is None
    else Path(TOKEN_FILE).expanduser()
).resolve()

git_exe = resolve_program("", "git")
conda_override = CONDA_EXE or os.environ.get("CONDA_EXE", "")
conda_exe = resolve_program(conda_override, "conda")
requested_backend = ENV_BACKEND.strip().lower()
if requested_backend not in {"auto", "current_python", "conda"}:
    raise ValueError("ENV_BACKEND 必须是 auto/current_python/conda")
environment_backend = (
    ("current_python" if is_kaggle or not conda_exe else "conda")
    if requested_backend == "auto"
    else requested_backend
)
python_exe = sys.executable

if not repo_url or repo_url.startswith("-"):
    raise ValueError("REPO_URL 不能为空或以 '-' 开头")
if not branch or branch.startswith("-"):
    raise ValueError("BRANCH 不能为空或以 '-' 开头")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", CONDA_ENV):
    raise ValueError("CONDA_ENV 只能包含字母、数字、点、下划线和连字符")
if target_dir in {Path("/"), Path.home().resolve(), Path("/kaggle/working")}:
    raise ValueError("TARGET_DIR 不能是根目录、用户主目录或整个 Kaggle working 目录")
if environment_backend == "conda" and not conda_exe:
    raise RuntimeError("选择了 Conda 后端但未找到 conda；可改 ENV_BACKEND='current_python'")
if TORCH_INDEX_URL:
    parsed_index = urlparse(TORCH_INDEX_URL)
    if parsed_index.scheme != "https" or not parsed_index.netloc:
        raise ValueError("TORCH_INDEX_URL 必须是完整 HTTPS URL")

network_checks = {}
if CHECK_NETWORK:
    network_checks = {
        "github": probe_url("https://github.com"),
        "pypi": probe_url("https://pypi.org/simple/"),
    }

torch_summary = {"installed": importlib.util.find_spec("torch") is not None}
if torch_summary["installed"]:
    import torch

    torch_summary.update({
        "version": torch.__version__,
        "cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count(),
        "devices": [
            torch.cuda.get_device_name(index)
            for index in range(torch.cuda.device_count())
        ],
    })

storage_probe_root = target_dir if target_dir.exists() else target_dir.parent
while not storage_probe_root.exists() and storage_probe_root != storage_probe_root.parent:
    storage_probe_root = storage_probe_root.parent
disk = shutil.disk_usage(storage_probe_root)
nvidia_smi = None
if shutil.which("nvidia-smi"):
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,driver_version,memory.total,memory.used,utilization.gpu",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    nvidia_smi = result.stdout.strip() or result.stderr.strip()

preflight = {
    "hostname": platform.node(),
    "platform": platform.platform(),
    "is_kaggle": is_kaggle,
    "environment_backend": environment_backend,
    "bootstrap_python": python_exe,
    "git": git_exe or None,
    "conda": conda_exe or None,
    "workspace_root": str(workspace_root),
    "storage_probe_root": str(storage_probe_root),
    "workspace_writable": storage_probe_root.is_dir() and os.access(storage_probe_root, os.W_OK),
    "target_dir": str(target_dir),
    "target_exists": target_dir.exists(),
    "disk_free_gib": round(disk.free / 1024**3, 2),
    "network": network_checks,
    "torch": torch_summary,
    "nvidia_smi": nvidia_smi,
    "mode": "execute" if RUN_SETUP else "preview",
}
print(json.dumps(preflight, indent=2, ensure_ascii=False))

if RUN_SETUP and not git_exe:
    raise RuntimeError("未找到 git")
if RUN_SETUP and not preflight["workspace_writable"]:
    raise RuntimeError(f"目标存储目录不可写: {storage_probe_root}")
if RUN_SETUP and disk.free < 3 * 1024**3:
    raise RuntimeError("可用磁盘少于 3 GiB")
if RUN_SETUP and CHECK_NETWORK and not all(item["ok"] for item in network_checks.values()):
    raise RuntimeError("GitHub/PyPI 网络检查失败；Kaggle 请在 Settings 中开启 Internet")
if (
    RUN_SETUP
    and is_kaggle
    and not ALLOW_CPU
    and torch_summary["installed"]
    and not torch_summary.get("cuda_available", False)
    and not REINSTALL_TORCH
):
    raise RuntimeError("Kaggle 当前 Torch 看不到 CUDA；请在 Settings 中选择 GPU 后重启 Session")

### 3. 一键执行

这是唯一会产生外部修改的主体。Kaggle/current-Python 后端不会创建 Conda 环境，也不会重装已经可用的 Torch。

In [ ]:
def run_command(
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    capture: bool = False,
) -> subprocess.CompletedProcess[str]:
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=cwd,
        env=env,
        check=True,
        text=True,
        capture_output=capture,
    )


def git_output(*arguments: str) -> str:
    result = run_command(
        [git_exe, "-C", str(target_dir), *arguments],
        capture=True,
    )
    return result.stdout.strip()


def environment_python_command() -> list[str]:
    if environment_backend == "conda":
        return [conda_exe, "run", "--name", CONDA_ENV, "python"]
    return [python_exe]


setup_summary = None
smoke_summary_path = None
smoke_archive_path = None

if not RUN_SETUP:
    print("预览模式：不会克隆、更新、安装或创建令牌。")
    print("启用方式：将 RUN_SETUP 改为 True，然后 Run All。")
else:
    if not target_dir.exists():
        target_dir.parent.mkdir(parents=True, exist_ok=True)
        run_command([
            git_exe,
            "clone",
            "--branch",
            branch,
            "--single-branch",
            repo_url,
            str(target_dir),
        ])
    else:
        if not (target_dir / ".git").is_dir():
            raise RuntimeError(f"目标目录已存在但不是 Git 仓库: {target_dir}")
        origin_url = git_output("remote", "get-url", "origin")
        if normalized_repo_url(origin_url) != normalized_repo_url(repo_url):
            raise RuntimeError(
                f"目标仓库 origin 不匹配；现有={origin_url!r}，期望={repo_url!r}"
            )
        current_branch = git_output("branch", "--show-current")
        if current_branch != branch:
            raise RuntimeError(
                f"目标仓库分支为 {current_branch!r}，期望 {branch!r}；不会自动切换"
            )
        if UPDATE_EXISTING:
            dirty = git_output("status", "--porcelain")
            if dirty:
                raise RuntimeError("已有仓库存在未提交修改，拒绝自动更新")
            run_command(
                [git_exe, "-C", str(target_dir), "fetch", "--prune", "origin", branch]
            )
            run_command(
                [git_exe, "-C", str(target_dir), "merge", "--ff-only", "FETCH_HEAD"]
            )
        else:
            print("复用已有仓库且不改变工作区；UPDATE_EXISTING=False。")

    if create_auth_token:
        token_file.parent.mkdir(parents=True, exist_ok=True)
        os.chmod(token_file.parent, stat.S_IRWXU)
        if token_file.is_symlink():
            raise RuntimeError(f"拒绝使用符号链接令牌文件: {token_file}")
        if token_file.exists() and not token_file.is_file():
            raise RuntimeError(f"令牌路径已存在但不是普通文件: {token_file}")
        if not token_file.exists():
            with token_file.open("x", encoding="utf-8") as handle:
                handle.write(secrets.token_urlsafe(32) + "\n")
        os.chmod(token_file, stat.S_IRUSR | stat.S_IWUSR)
        token_value = token_file.read_text(encoding="utf-8").strip()
        if not token_value:
            raise RuntimeError(f"认证令牌文件为空: {token_file}")
        token_fingerprint = hashlib.sha256(token_value.encode("utf-8")).hexdigest()[:12]
    else:
        token_fingerprint = None

    if environment_backend == "conda":
        bootstrap_script = target_dir / "scripts/remote/bootstrap_learner_env.sh"
        if not bootstrap_script.is_file():
            raise FileNotFoundError(bootstrap_script)
        bootstrap_command = [
            "bash",
            str(bootstrap_script),
            "--env-name",
            CONDA_ENV,
        ]
        if TORCH_INDEX_URL:
            bootstrap_command.extend(["--torch-index-url", TORCH_INDEX_URL])
        if REINSTALL_TORCH:
            bootstrap_command.append("--reinstall-torch")
        if ALLOW_CPU:
            bootstrap_command.append("--allow-cpu")
        child_env = os.environ.copy()
        child_env["HKRL_CONDA_BIN"] = conda_exe
        run_command(bootstrap_command, cwd=target_dir, env=child_env)
    else:
        if REINSTALL_TORCH or not torch_summary["installed"]:
            torch_command = [
                python_exe,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "torch",
            ]
            if REINSTALL_TORCH:
                torch_command.append("--force-reinstall")
            if TORCH_INDEX_URL:
                torch_command.extend(["--index-url", TORCH_INDEX_URL])
            run_command(torch_command)
        else:
            print("复用当前 kernel 已安装的 PyTorch，不重装。")
        run_command([
            python_exe,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{target_dir}/python[dev,logging,distributed]",
        ])

    validation_code = f"""
import json
import torch
from hkrl.utils.config import load_train_config
cfg = load_train_config('configs/train/ssh_remote_learner.yaml')
summary = {{
    'torch': torch.__version__,
    'cuda_build': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device_count': torch.cuda.device_count(),
    'cuda_devices': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'algorithm': cfg.algorithm,
    'model': cfg.model.name,
    'learner_device': cfg.learner.device,
}}
print(json.dumps(summary, sort_keys=True))
if not {ALLOW_CPU!r} and not torch.cuda.is_available():
    raise SystemExit('CUDA is required but unavailable')
""".strip()
    run_command(
        [*environment_python_command(), "-c", validation_code],
        cwd=target_dir,
    )

    if RUN_KAGGLE_SMOKE and is_kaggle:
        run_id = (
            datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            + "-"
            + secrets.token_hex(3)
        )
        smoke_root = target_dir / "runs" / f"kaggle-smoke-{run_id}"
        learner_root = smoke_root / "learner-update"
        phase8_root = smoke_root / "phase8"
        synthetic_batch = learner_root / "batches" / "synthetic_v000000.npz"
        learner_summary_path = learner_root / "summary.json"
        phase8_summary_path = phase8_root / "summary.json"
        smoke_summary_path = target_dir / "runs" / "kaggle-smoke-summary.json"
        smoke_config = (
            "configs/train/remote_learner.yaml"
            if ALLOW_CPU
            else "configs/train/ssh_remote_learner.yaml"
        )
        synthetic_batch_code = """
from pathlib import Path
import sys
import numpy as np
from hkrl.models.heads import ACTION_TENSOR_DIM_NO_MACRO
from hkrl.spaces import action_mask_layout, make_observation_space
from hkrl.training.batch_io import save_rollout_batch
from hkrl.training.rollout_buffer import RolloutBatch
from hkrl.utils.config import load_task_config

output = Path(sys.argv[1])
task = load_task_config(sys.argv[2])
space = make_observation_space(task.observation.max_entities, task.observation.tier)
time_steps, num_envs = 4, 1
enable_macro = task.action.enable_macro_actions
action_dim = ACTION_TENSOR_DIM_NO_MACRO + int(enable_macro)
mask_dim = len(action_mask_layout(enable_macro, task.action.n_macro_actions))
actions = np.zeros((time_steps, num_envs, action_dim), dtype=np.int64)
actions[:, :, 0:2] = 1
entity_mask = np.zeros((time_steps, num_envs, *space['entity_mask'].shape), dtype=bool)
entity_mask[:, :, 0] = True
signal = np.arange(1, time_steps + 1, dtype=np.float32).reshape(time_steps, num_envs)
batch = RolloutBatch(
    obs_global=np.zeros((time_steps, num_envs, *space['global'].shape), dtype=np.float32),
    obs_player=np.zeros((time_steps, num_envs, *space['player'].shape), dtype=np.float32),
    obs_entities=np.zeros((time_steps, num_envs, *space['entities'].shape), dtype=np.float32),
    entity_mask=entity_mask,
    actions=actions,
    log_probs=np.full((time_steps, num_envs), -1.0, dtype=np.float32),
    values=np.zeros((time_steps, num_envs), dtype=np.float32),
    advantages=signal.copy(),
    returns=signal.copy(),
    rewards=np.ones((time_steps, num_envs), dtype=np.float32),
    dones=np.array([[False], [False], [False], [True]], dtype=bool),
    truncateds=np.zeros((time_steps, num_envs), dtype=bool),
    action_masks=np.ones((time_steps, num_envs, mask_dim), dtype=bool),
    prev_actions=np.zeros((time_steps, num_envs, action_dim), dtype=np.int64),
    prev_rewards=np.zeros((time_steps, num_envs), dtype=np.float32),
    rnn_states=None,
    episode_ids=np.ones((time_steps, num_envs), dtype=np.uint64),
    task_ids=np.full((time_steps, num_envs), task.wire_id, dtype=np.int64),
    policy_version=0,
)
save_rollout_batch(output, batch)
print(output)
""".strip()
        run_command([
            *environment_python_command(),
            "-c",
            synthetic_batch_code,
            str(synthetic_batch),
            "configs/tasks/gruz_mother.yaml",
        ], cwd=target_dir)
        learner_result = run_command([
            *environment_python_command(),
            "scripts/run_learner.py",
            "--config",
            smoke_config,
            "--tasks",
            "configs/tasks/gruz_mother.yaml",
            "--bind",
            "127.0.0.1:0",
            "--batch-dir",
            str(synthetic_batch.parent),
            "--checkpoint-dir",
            str(learner_root / "checkpoints"),
            "--publish-every-updates",
            "1",
        ], cwd=target_dir, capture=True)
        if learner_result.stdout:
            print(learner_result.stdout, end="")
        if learner_result.stderr:
            print(learner_result.stderr, file=sys.stderr, end="")
        learner_lines = [
            line for line in learner_result.stdout.splitlines() if line.strip()
        ]
        if not learner_lines:
            raise RuntimeError("Learner smoke 没有输出 JSON")
        learner_summary = json.loads(learner_lines[-1])
        if (
            learner_summary.get("accepted_batches") != 1
            or learner_summary.get("submitted_batches") != 1
            or learner_summary.get("policy_version") != 1
        ):
            raise RuntimeError(f"合成 GPU 训练更新失败: {learner_summary}")
        if (
            not ALLOW_CPU
            and not str(learner_summary.get("device", "")).startswith("cuda")
        ):
            raise RuntimeError(f"Learner 未在 CUDA 上运行: {learner_summary}")
        learner_summary_path.parent.mkdir(parents=True, exist_ok=True)
        learner_summary_path.write_text(
            json.dumps(learner_summary, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        run_command([
            *environment_python_command(),
            "scripts/run_phase8_smoke.py",
            "--config",
            smoke_config,
            "--tasks",
            "configs/tasks/gruz_mother.yaml",
            "--work-dir",
            str(phase8_root),
            "--num-workers",
            "1",
            "--output",
            str(phase8_summary_path),
        ], cwd=target_dir)
        phase8_summary = json.loads(phase8_summary_path.read_text(encoding="utf-8"))
        if phase8_summary.get("ok") is not True:
            raise RuntimeError(f"Phase 8 smoke 失败: {phase8_summary}")
        combined_smoke_summary = {
            "ok": True,
            "synthetic_train_update": True,
            "learner": learner_summary,
            "phase8": phase8_summary,
            "artifacts": {
                "run_root": str(smoke_root),
                "synthetic_batch": str(synthetic_batch),
                "learner_summary": str(learner_summary_path),
                "phase8_summary": str(phase8_summary_path),
            },
        }
        smoke_summary_path.write_text(
            json.dumps(combined_smoke_summary, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        smoke_archive = shutil.make_archive(
            str(target_dir / "runs" / "kaggle-smoke-artifacts"),
            "zip",
            root_dir=smoke_root,
        )
        smoke_archive_path = Path(smoke_archive)

    if RUN_PROJECT_TESTS:
        if environment_backend == "conda":
            test_command = [conda_exe, "run", "--name", CONDA_ENV, "make", "check"]
            test_cwd = target_dir
        else:
            test_command = [python_exe, "-m", "pytest", "-q"]
            test_cwd = target_dir / "python"
        run_command(test_command, cwd=test_cwd)

    commit = git_output("rev-parse", "HEAD")
    setup_summary = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "repo_url": repo_url,
        "branch": branch,
        "commit": commit,
        "target_dir": str(target_dir),
        "is_kaggle": is_kaggle,
        "environment_backend": environment_backend,
        "python_exe": python_exe,
        "conda_env": CONDA_ENV if environment_backend == "conda" else None,
        "conda_exe": conda_exe if environment_backend == "conda" else None,
        "allow_cpu": ALLOW_CPU,
        "token_file": str(token_file) if create_auth_token else None,
        "token_fingerprint": token_fingerprint,
        "smoke_summary": None if smoke_summary_path is None else str(smoke_summary_path),
        "smoke_archive": None if smoke_archive_path is None else str(smoke_archive_path),
    }
    manifest = target_dir / "runs" / "setup" / "clone_setup_summary.json"
    manifest.parent.mkdir(parents=True, exist_ok=True)
    manifest.write_text(
        json.dumps(setup_summary, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print("安装完成：")
    print(json.dumps(setup_summary, indent=2, ensure_ascii=False))
    print(f"安装摘要：{manifest}")
    if environment_backend == "current_python":
        print("建议在继续交互导入 HKRL 前重启当前 Notebook Session。")

## Checks

### 4. 检查仓库、安装摘要和 Kaggle smoke

本格只读取状态，不联网、不修改。Kaggle smoke 的 `ok=true` 和 `synthetic_train_update=true` 代表 GPU 训练更新与离线分布式结构通过，不代表真实游戏能力。

In [ ]:
if not target_dir.exists():
    print(f"目标目录尚不存在：{target_dir}")
else:
    checks = {
        "target_dir": str(target_dir),
        "is_git_repo": (target_dir / ".git").is_dir(),
        "training_notebook_exists": (
            target_dir / "notebooks" / "remote_gpu_training.ipynb"
        ).is_file(),
        "bootstrap_exists": (
            target_dir / "scripts" / "remote" / "bootstrap_learner_env.sh"
        ).is_file(),
    }
    manifest = target_dir / "runs" / "setup" / "clone_setup_summary.json"
    smoke_summary = target_dir / "runs" / "kaggle-smoke-summary.json"
    smoke_archive = target_dir / "runs" / "kaggle-smoke-artifacts.zip"
    if manifest.is_file():
        checks["manifest"] = json.loads(manifest.read_text(encoding="utf-8"))
    if smoke_summary.is_file():
        checks["kaggle_smoke"] = json.loads(smoke_summary.read_text(encoding="utf-8"))
    checks["kaggle_smoke_archive"] = (
        str(smoke_archive) if smoke_archive.is_file() else None
    )
    print(json.dumps(checks, indent=2, ensure_ascii=False))

## Next Steps

### Kaggle

成功标志：预检显示 `is_kaggle=true`、`environment_backend=current_python`、`cuda_available=true`，并且 `runs/kaggle-smoke-summary.json` 中 `ok=true`、`synthetic_train_update=true`、`learner.device=cuda`、`learner.accepted_batches=1`、`learner.policy_version=1`。

下载或保存以下无密钥产物：

```text
/kaggle/working/hk_Rl/runs/setup/clone_setup_summary.json
/kaggle/working/hk_Rl/runs/kaggle-smoke-summary.json
/kaggle/working/hk_Rl/runs/kaggle-smoke-artifacts.zip
```

Kaggle smoke 完成后不要启动 `start_learner_stack.sh` 等待 Windows；Kaggle 会话不是这里设计的 SSH 长连训练服务器。若要进一步测试更新逻辑，可以把已采集的、版本兼容的 rollout NPZ 作为 Kaggle Dataset 输入，再用 `run_learner.py --batch-dir` 做有限离线实验。
建议单独上传并打开 `notebooks/kaggle_training.ipynb` 完成该离线训练轮次；它会校验 NPZ、恢复 checkpoint、执行一次更新并打包下一版模型。

### 普通 SSH GPU 训练机

```bash
cd ~/hk_Rl
export HKRL_AUTH_TOKEN="$(< ~/.config/hkrl/auth_token)"
conda run --name hkrl-learner jupyter lab --no-browser
```

然后打开 `notebooks/remote_gpu_training.ipynb`，先运行 `inspect`，再切换到 `train`。不要把 Jupyter、Learner 或 checkpoint 端口直接暴露到公网。